In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
# ------------------------------------------------------------
# OPTION B: One CSV per policy
# ------------------------------------------------------------
POLICY_FILES = {
     "CQL": Path("../outputs/cql_test_results2.csv"),
     "Random": Path("../outputs/trajectories_random_5k.csv"),
     "Expert": Path("../outputs/trajectories_expert_5k.csv"),
 }

REFERENCE_POLICY = "RANDOM"  # policy used on the x-axis in pairwise plots

In [12]:
# Column names in your CSV. Adjust only this dictionary if needed.
COL = {
    "policy": "policy",
    "query_id": "query_id",
    "step": "step",
    "reward": "reward",
    "delta_ndcg": "delta_ndcg",
    "ndcg_before": "ndcg_before",  # optional
    "ndcg_after": "ndcg_after",      # optional
    "cumulative_cost": "cumulative_cost",
    "action_cost": "cost",           # optional fallback
    "elapsed_ms": "elapsed_ms",  # optional fallback
    "terminal": "terminal",          # optional
    "timeout": "timeout",            # optional
}

In [13]:
import numpy as np
import pandas as pd


def _to_bool(series: pd.Series) -> pd.Series:
    """Convert bool, 0/1, and common CSV string representations to bool."""
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y"})
    )


def build_trajectory_summary(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Reduce one policy's action-level rollout CSV to one row per query_id.

    Required columns:
        query_id, step, reward, delta_ndcg, terminal, timeout

    Recommended cumulative columns:
        cum_cost, cum_latency

    Fallback action-level columns:
        cost, elapsed_ms

    Endpoint:
        The final row whose terminal OR timeout flag is True.

    Output:
        query_id, final_step, ended_by, n_actions, total_reward,
        delta_ndcg, cum_cost, cum_latency
    """
    required = [
        "query_id",
        "step",
        "reward",
        "delta_ndcg",
        "terminal",
        "timeout",
    ]
    missing = [column for column in required if column not in raw.columns]
    if missing:
        raise ValueError(f"Missing required CSV columns: {missing}")

    df = raw.copy()

    # Make numeric calculations safe.
    numeric_columns = [
        "step",
        "reward",
        "delta_ndcg",
        "cumulative_cost",
        "cost",
        "elapsed_ms",
        "ndcg_before",
        "ndcg_after",
    ]

    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    if df["query_id"].isna().any():
        raise ValueError("Found rows with a missing query_id.")

    if df["step"].isna().any():
        raise ValueError("Found rows with a missing or invalid step value.")

    # A row is the trajectory endpoint if it is terminal or timed out.
    df["_terminal"] = _to_bool(df["terminal"])
    df["_timeout"] = _to_bool(df["timeout"])
    df["_endpoint"] = df["_terminal"] | df["_timeout"]

    # Sort before selecting the final marked endpoint for each query.
    df = df.sort_values(["query_id", "step"]).reset_index(drop=True)

    endpoint_rows = (
        df[df["_endpoint"]]
        .groupby("query_id", group_keys=False)
        .tail(1)
        .copy()
    )

    # Do not hide incomplete rollouts.
    all_query_ids = set(df["query_id"].unique())
    ended_query_ids = set(endpoint_rows["query_id"].unique())
    missing_endpoints = sorted(all_query_ids - ended_query_ids)

    if missing_endpoints:
        preview = missing_endpoints[:10]
        raise ValueError(
            f"{len(missing_endpoints)} query trajectories have no terminal or "
            f"timeout row. Example query_ids: {preview}"
        )

    # Total reward and total NDCG improvement are trajectory-level sums.
    trajectory_totals = (
        df.groupby("query_id", as_index=False)
        .agg(
            n_actions=("step", "size"),
            total_reward=("reward", "sum"),
            delta_ndcg_sum=("delta_ndcg", "sum"),
        )
    )

    summary = endpoint_rows[
        ["query_id", "step", "_terminal", "_timeout"]
    ].rename(columns={"step": "final_step"})

    summary["ended_by"] = np.select(
        [
            summary["_terminal"] & summary["_timeout"],
            summary["_terminal"],
            summary["_timeout"],
        ],
        [
            "terminal_and_timeout",
            "terminal",
            "timeout",
        ],
        default="unknown",
    )

    summary = summary.drop(columns=["_terminal", "_timeout"])
    summary = summary.merge(trajectory_totals, on="query_id", how="left", validate="one_to_one")

    # If the CSV exports absolute values, use final NDCG - initial NDCG.
    # Otherwise, sum per-step delta_ndcg values.
    if {"ndcg_before", "ndcg_after"}.issubset(df.columns):
        ndcg_at_endpoint = endpoint_rows[
            ["query_id", "ndcg_before", "ndcg_after"]
        ].copy()

        ndcg_at_endpoint["delta_ndcg"] = (
            ndcg_at_endpoint["ndcg_after"] - ndcg_at_endpoint["ndcg_before"]
        )

        summary = summary.merge(
            ndcg_at_endpoint[["query_id", "delta_ndcg"]],
            on="query_id",
            how="left",
            validate="one_to_one"
        )

        summary["delta_ndcg"] = summary["delta_ndcg"].fillna(
            summary["delta_ndcg_sum"]
        )
    else:
        summary["delta_ndcg"] = summary["delta_ndcg_sum"]

    summary = summary.drop(columns=["delta_ndcg_sum"])

    # Prefer cumulative metrics taken from the endpoint row.
    if "cumulative_cost" in endpoint_rows.columns:
        summary = summary.merge(
            endpoint_rows[["query_id", "cumulative_cost"]],
            on="query_id",
            how="left",
            validate="one_to_one"
        )
    elif "cost" in df.columns:
        costs = (
            df.groupby("query_id", as_index=False)["cost"]
            .sum()
            .rename(columns={"cost": "cum_cost"})
        )
        summary = summary.merge(costs, on="query_id", how="left", validate="one_to_one")
    else:
        summary["cum_cost"] = np.nan

    if "cum_latency" in endpoint_rows.columns:
        summary = summary.merge(
            endpoint_rows[["query_id", "cum_latency"]],
            on="query_id",
            how="left",
            validate="one_to_one"
        )
    elif "elapsed_ms" in df.columns:
        latencies = (
            df.groupby("query_id", as_index=False)["elapsed_ms"]
            .sum()
            .rename(columns={"elapsed_ms": "cum_latency"})
        )
        summary = summary.merge(latencies, on="query_id", how="left", validate="one_to_one")
    else:
        summary["cum_latency"] = np.nan

    return summary.sort_values("query_id").reset_index(drop=True)

In [14]:
# One trajectory-level DataFrame per policy.
policy_summaries = {}

for policy_name, csv_path in POLICY_FILES.items():
    raw_policy_rows = pd.read_csv(csv_path)

    # terminal/timeout are used here to choose each query's endpoint.
    policy_summaries[policy_name] = build_trajectory_summary(raw_policy_rows)

    print(
        f"{policy_name}: "
        f"{len(policy_summaries[policy_name])} completed query trajectories"
    )

if REFERENCE_POLICY not in policy_summaries:
    raise ValueError(
        f"REFERENCE_POLICY='{REFERENCE_POLICY}' is not in POLICY_FILES. "
        f"Available: {list(policy_summaries)}"
    )

CQL: 50 completed query trajectories


ValueError: Missing required CSV columns: ['delta_ndcg']

In [ ]:
def pair_policy_summaries(
    reference_summary: pd.DataFrame,
    candidate_summary: pd.DataFrame,
    reference_name: str,
    candidate_name: str,
) -> pd.DataFrame:
    """
    Pair two trajectory summaries by query_id.

    Expected summary columns from build_trajectory_summary:
        query_id, final_step, ended_by, n_actions, total_reward,
        delta_ndcg, cum_cost, cum_latency
    """
    compare_columns = [
        "query_id",
        "delta_ndcg",
        "total_reward",
        "cum_cost",
        "cum_latency",
        "n_actions",
        "ended_by",
    ]

    required = set(compare_columns)
    for name, summary in [
        (reference_name, reference_summary),
        (candidate_name, candidate_summary),
    ]:
        missing = required - set(summary.columns)
        if missing:
            raise ValueError(
                f"{name} summary is missing required columns: {sorted(missing)}"
            )

    reference = reference_summary[compare_columns].rename(
        columns={
            column: f"{reference_name}_{column}"
            for column in compare_columns
            if column != "query_id"
        }
    )

    candidate = candidate_summary[compare_columns].rename(
        columns={
            column: f"{candidate_name}_{column}"
            for column in compare_columns
            if column != "query_id"
        }
    )

    paired = reference.merge(
        candidate,
        on="query_id",
        how="inner",
        validate="one_to_one",
    )

    paired["delta_ndcg_difference"] = (
        paired[f"{candidate_name}_delta_ndcg"]
        - paired[f"{reference_name}_delta_ndcg"]
    )

    paired["reward_difference"] = (
        paired[f"{candidate_name}_total_reward"]
        - paired[f"{reference_name}_total_reward"]
    )

    paired["cost_difference"] = (
        paired[f"{candidate_name}_cum_cost"]
        - paired[f"{reference_name}_cum_cost"]
    )

    paired["latency_difference"] = (
        paired[f"{candidate_name}_cum_latency"]
        - paired[f"{reference_name}_cum_latency"]
    )

    return paired

In [ ]:
def plot_paired_ndcg(
    reference_summary: pd.DataFrame,
    candidate_summary: pd.DataFrame,
    reference_name: str,
    candidate_name: str,
) -> pd.DataFrame:
    paired = pair_policy_summaries(
        reference_summary=reference_summary,
        candidate_summary=candidate_summary,
        reference_name=reference_name,
        candidate_name=candidate_name,
    )

    x_col = f"{reference_name}_delta_ndcg"
    y_col = f"{candidate_name}_delta_ndcg"

    paired = paired.dropna(subset=[x_col, y_col]).copy()

    if paired.empty:
        raise ValueError("No matched queries with valid delta_ndcg values.")

    candidate_wins = paired[y_col] > paired[x_col]
    point_colors = np.where(candidate_wins, "#2a9d8f", "#e76f51")

    low = min(paired[x_col].min(), paired[y_col].min())
    high = max(paired[x_col].max(), paired[y_col].max())
    padding = max((high - low) * 0.05, 0.01)

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.scatter(
        paired[x_col],
        paired[y_col],
        c=point_colors,
        alpha=0.70,
        s=48,
        edgecolor="white",
        linewidth=0.5,
    )

    ax.plot(
        [low - padding, high + padding],
        [low - padding, high + padding],
        linestyle="--",
        color="black",
        linewidth=1,
        label="Equal ΔNDCG",
    )

    win_rate = candidate_wins.mean()
    mean_difference = paired["delta_ndcg_difference"].mean()

    ax.set_xlim(low - padding, high + padding)
    ax.set_ylim(low - padding, high + padding)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(
        f"Final ΔNDCG per query\n"
        f"{candidate_name} vs {reference_name} "
        f"({len(paired)} matched queries)"
    )
    ax.set_xlabel(f"{reference_name}: final ΔNDCG")
    ax.set_ylabel(f"{candidate_name}: final ΔNDCG")

    ax.text(
        0.03,
        0.97,
        f"{candidate_name} win rate: {win_rate:.1%}\n"
        f"Mean ΔNDCG difference: {mean_difference:.4f}",
        transform=ax.transAxes,
        verticalalignment="top",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85},
    )

    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    return paired

In [ ]:
def plot_paired_reward(
    reference_summary: pd.DataFrame,
    candidate_summary: pd.DataFrame,
    reference_name: str,
    candidate_name: str,
) -> pd.DataFrame:
    paired = pair_policy_summaries(
        reference_summary=reference_summary,
        candidate_summary=candidate_summary,
        reference_name=reference_name,
        candidate_name=candidate_name,
    )

    x_col = f"{reference_name}_total_reward"
    y_col = f"{candidate_name}_total_reward"

    paired = paired.dropna(subset=[x_col, y_col]).copy()

    if paired.empty:
        raise ValueError("No matched queries with valid total_reward values.")

    candidate_wins = paired[y_col] > paired[x_col]
    point_colors = np.where(candidate_wins, "#2a9d8f", "#e76f51")

    low = min(paired[x_col].min(), paired[y_col].min())
    high = max(paired[x_col].max(), paired[y_col].max())
    padding = max((high - low) * 0.05, 0.01)

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.scatter(
        paired[x_col],
        paired[y_col],
        c=point_colors,
        alpha=0.70,
        s=48,
        edgecolor="white",
        linewidth=0.5,
    )

    ax.plot(
        [low - padding, high + padding],
        [low - padding, high + padding],
        linestyle="--",
        color="black",
        linewidth=1,
        label="Equal cumulative reward",
    )

    win_rate = candidate_wins.mean()
    mean_difference = paired["reward_difference"].mean()

    ax.set_xlim(low - padding, high + padding)
    ax.set_ylim(low - padding, high + padding)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(
        f"Cumulative reward per query\n"
        f"{candidate_name} vs {reference_name} "
        f"({len(paired)} matched queries)"
    )
    ax.set_xlabel(f"{reference_name}: cumulative reward")
    ax.set_ylabel(f"{candidate_name}: cumulative reward")

    ax.text(
        0.03,
        0.97,
        f"{candidate_name} win rate: {win_rate:.1%}\n"
        f"Mean reward difference: {mean_difference:.4f}",
        transform=ax.transAxes,
        verticalalignment="top",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85},
    )

    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    return paired